# Colab 06 · Image model: produce (rotten vs fresh)

**Business question:** is the fruit rotten?

**Goal:** train and compare two models on the produce photos:
- **small CNN from scratch** (course architecture: Conv2D, MaxPooling, Dropout, Flatten, Dense softmax);
- **MobileNetV2 transfer learning** (ImageNet weights): base frozen first, then the top 30 layers fine-tuned. This one goes to the website.

Both use augmentation (flip, rotation, zoom, brightness, contrast), class weights and early stopping. The decision
threshold is tuned on the validation set so that at least 90% of rotten photos are caught. Grad-CAM shows where
the model looks.

**Runtime:** GPU (Runtime → Change runtime type → T4 GPU). About 15–30 minutes.

**Inputs (Google Drive):** `MyDrive/rrf/code.zip`, `MyDrive/rrf/images.zip`
**Outputs:** copied to `MyDrive/rrf/` (metrics, predictions, figures, and the trained MobileNetV2 model).

In [ ]:
# ---- Setup: GPU check, Google Drive, code, packages ----
import os, subprocess, sys
from pathlib import Path

gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
assert gpu, "No GPU! Menu: Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", gpu)

from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/rrf")          # project folder on your Drive
REPO = Path("/content/application-of-text-qualification")

# Get the project code. Two ways, because the GitHub repo is private:
#  A) code.zip on Drive (MyDrive/rrf/code.zip, made with `python scripts/make_code_zip.py`): no token needed
#  B) git clone with a token saved in Colab "Secrets" (key icon, left sidebar) as GITHUB_TOKEN
CODE_ZIP = DRIVE / "code.zip"
if CODE_ZIP.exists():
    import shutil, zipfile
    shutil.rmtree(REPO, ignore_errors=True)
    zipfile.ZipFile(CODE_ZIP).extractall(REPO)
    print("code from", CODE_ZIP)
elif not REPO.exists():
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q",
                    f"https://{token}@github.com/Flaxxxy/application-of-text-qualification.git", str(REPO)],
                   check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("ready:", REPO)

## 1. Get the images from Drive

In [ ]:
import zipfile
zipfile.ZipFile(DRIVE / "images.zip").extractall(REPO)
import pandas as pd
m = pd.read_csv(REPO / "data/processed/images/produce_manifest.csv")
pd.crosstab(m["label"], m["split"], margins=True)

## 2. Small CNN from scratch (baseline)

In [ ]:
from src.config import set_seed
from src.image_models import run_experiment
set_seed()
cnn = run_experiment("produce", "small_cnn")

## 3. MobileNetV2 transfer learning

In [ ]:
mnv2 = run_experiment("produce", "mobilenetv2")

## 4. Comparison (test set, tuned threshold)

In [ ]:
import pandas as pd
rows = {}
for name, mtr in [("small_cnn", cnn), ("mobilenetv2", mnv2)]:
    t = mtr["test"]; bad = t["per_class"]["rotten"]
    rows[name] = {"accuracy": t["accuracy"], "macro-F1": t["macro_f1"], "recall rotten": bad["recall"],
                  "precision rotten": bad["precision"], "threshold": mtr["threshold"],
                  "macro-F1 at 0.5": mtr["test_default"]["macro_f1"], "size MB": mtr["size_mb"]}
display(pd.DataFrame(rows).T)
display(pd.DataFrame({n: {g: v["macro_f1"] for g, v in mtr["test_per_group"].items()}
                      for n, mtr in [("small_cnn", cnn), ("mobilenetv2", mnv2)]}))

## 5. Copy results to Drive and list what to bring back

In [ ]:
import shutil
files = []
for name in ["small_cnn", "mobilenetv2"]:
    files += [f"results/colab/metrics_image_produce__{name}.json",
              f"results/predictions/image_produce_{name}_test.csv",
              f"results/figures/cm_image_produce_{name}.png",
              f"results/figures/history_produce_{name}.png",
              f"results/figures/gradcam_produce_{name}.png"]
files.append("models/train/image_produce_mobilenetv2.keras")   # needed for the website (Phase 5)
for f in files:
    dst = DRIVE / f
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(REPO / f, dst)

print("Copy these files from Google Drive (MyDrive/rrf/...) into the SAME paths in the project:\n")
for f in files:
    print("  ", f)

## Key findings

*(Filled in after the run.)*